This Jupyter Notebook contains code to do the following:
- ff

The notebook is divided into sections and each code block is suplemented with comments and markdown text to guide the user about whats going on.

All or parts of the code from this notebook can be easily copied to create standalone python script files.
Further information about the methods used in this notebook can be found in the following papers:
- https://arxiv.org/abs/2511.12685
- 


# Initialization

In [104]:
# importing necessary libraries
import os
import sys
import re
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from src.configManager import ConfigManager
import xarray as xr

config = ConfigManager()

In [105]:
# code to set up matplotlib and seaborn styles

plt.style.use('default')

# Save the default rcParams to revert back if needed
dflt_rcParams = mpl.rcParams.copy()

# Change the default font
mpl.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial']})

# Increase the font size for better readability
mpl.rcParams.update({'font.size': 14, 
                     'axes.titlesize': 14,
                     'axes.titleweight': 'bold', 
                     'axes.labelsize': 12,
                     'axes.labelweight': 'bold',
                     'xtick.labelsize': 12, 
                     'ytick.labelsize': 12, 
                     'legend.fontsize': 11})

# Remove the top and right spines for a cleaner look
mpl.rcParams.update({'axes.spines.right': False, 
                     'axes.spines.top': False,
                     'axes.spines.left': True,
                     'axes.spines.bottom': True})

# Add space between the plot and the title
mpl.rcParams.update({'axes.titlepad': 15})

# Set the figure size for all plots
mpl.rcParams.update({'figure.figsize': (10, 6)})

# Set the style for seaborn
sns.set_style("white")  # other options: "dark", "whitegrid", "darkgrid", "ticks"

In [106]:
state = config.get('processing_parameters')['state']
county = config.get('processing_parameters')['county']
start_year = config.get('processing_parameters')['start_year']
end_year = config.get('processing_parameters')['end_year']

ds_merged = xr.open_dataset(os.path.join(
    config.get("data_paths.merged_data_dir"),
    state,
    config.get("file_patterns.merged_file_pattern").format(
        state=state,
        county=county,
        start=start_year,
        end=end_year
    )
))  

# Exploring the Merged Data

In [107]:
# check the different details of the dataset in an interactive viewer
ds_merged

<xarray.Dataset> Size: 140MB
Dimensions:              (time: 385728, station: 17)
Coordinates:
  * time                 (time) datetime64[ns] 3MB 2014-01-01 ... 2024-12-31T...
  * station              (station) <U3 204B 'AVX' 'BUR' 'CQT' ... 'WHP' 'WJF'
    lat                  (station) <U7 476B ...
    lon                  (station) <U9 612B ...
Data variables:
    tmpf                 (time, station) float32 26MB ...
    sknt                 (time, station) float32 26MB ...
    gust                 (time, station) float32 26MB ...
    p01i                 (time, station) float32 26MB ...
    poccurence           (time, station) float32 26MB ...
    customers_out        (time) int64 3MB ...
    event_number_eaglei  (time) int64 3MB ...
Attributes:
    title:                   Merged Weather and Outage Data for Los Angeles, ...
    description:             This dataset contains weather data from multiple...
    state:                   California
    county:                  Los Angeles
    start_year:              2014
    end_year:                2024
    creation_date:           2026-01-21 11:15:39
    temporal_resolution:     15 minutes
    county_fips_code:        6037
    total_county_customers:  3799750

In [108]:
# get the timeseries range of the merged dataset
print(ds_merged['time'].values[0], ds_merged['time'].values[-1])

2014-01-01T00:00:00.000000000 2024-12-31T23:45:00.000000000


In [109]:
# get all the variables in the dataset
print(ds_merged.data_vars)

Data variables:
    tmpf                 (time, station) float32 26MB ...
    sknt                 (time, station) float32 26MB ...
    gust                 (time, station) float32 26MB ...
    p01i                 (time, station) float32 26MB ...
    poccurence           (time, station) float32 26MB ...
    customers_out        (time) int64 3MB ...
    event_number_eaglei  (time) int64 3MB ...


In [110]:
# get all the stations available in the merged dataset
print(ds_merged['station'].values)

['AVX' 'BUR' 'CQT' 'EMT' 'GXA' 'HHR' 'LAX' 'LGB' 'NUC' 'PMD' 'POC' 'SDB'
 'SMO' 'TOA' 'VNY' 'WHP' 'WJF']


In [ ]:
# get all the data for a specific station, e.g., 'MIA'
station='CLM'
ds_merged.sel(station=station)

In [ ]:
# select all values of a particular weather variable at a specific time (for all stations in the dataset in order)
print(ds_merged['tmpf'].sel(time='2020-06-15 12:00:00').values)

In [ ]:
# select a particular weather variable for a specific station at a specific time
print(ds_merged['tmpf'].sel(station=station, time='2020-06-15 12:00:00').values)

In [ ]:
ds_merged['customers_out'].sel(time='2020-06-15 12:00:00').values

In [ ]:
# Select time range and get all data
data = ds_merged.sel(time=slice('2021-06-27', '2021-06-28'))

# Access weather from specific station
station_weather = data.sel(station=station)

# Access outage data for the time period
outages = data['customers_out']

In [ ]:
outages.values

# Analyzing Events in the EAGLE-I Outage Data

In [ ]:
# create a dataframe of customers_out and event_number_eaglei from the xarray dataset
outage_df = ds_merged[['customers_out', 'event_number_eaglei']].to_dataframe().reset_index()
print(f"Size of the outage dataframe: {outage_df.shape}")
# keep only rows where customers_out > 0
outage_df = outage_df[outage_df['customers_out'] > 0]
print(f"Size of the outage dataframe after filtering customers_out > 0: {outage_df.shape}")

In [ ]:
outage_df.head()

In [ ]:
# Total number of outage events
print(f"Total number of outage events: {outage_df['event_number_eaglei'].nunique()}")
# Total number of outage events, excluding the ones that don't meet the customers_out threshold
customers_out_threshold = config.get('data_cleaning_parameters')['events_customer_threshold']
print(f"Total number of outage events (customers_out > {customers_out_threshold}): {outage_df[outage_df['customers_out'] >= customers_out_threshold]['event_number_eaglei'].nunique()}")

In [ ]:
# Size of the top 10 longest outage events (longest in terms of number of records)
# Since each record represents a 15-minute interval, it is a proxy for duration
print(outage_df.groupby('event_number_eaglei')
                          .size()
                          .sort_values(ascending=False)
                          .head(10))

In [ ]:
from src.eaglei_modules.eagleiEventProcessing import plot_eaglei_event_curves
plot_eaglei_event_curves(outage_df, event_number=616, event_method='eaglei', timestamp_column='time')

# Analyzing Weather Data

In [ ]:
# plot a time series of temperature for all the stations for a given time period
time_from = '2021-06-26 00:15:00'
time_to = '2021-07-01 05:30:00'
fig, ax = plt.subplots(figsize=(12, 5))
p = ds_merged['tmpf'].sel(time=slice(time_from, time_to), station=ds_merged['station']).plot.line(x='time', hue='station', marker='x', ax=ax)
ax.set_title(f'Temperature Time Series for All Stations ({time_from} to {time_to})')
ax.set_ylabel('Temperature (°F)')
ax.set_xlabel('Time')
fig.tight_layout()
plt.show()

# plot a time series of maximum temperature across all the stations for a given time period
fig, ax = plt.subplots(figsize=(12, 5))
p = ds_merged['tmpf'].sel(time=slice(time_from, time_to), station=ds_merged['station']).max(dim='station').plot.line(x='time', marker='x')
ax.set_title(f'Maximum Temperature Across All Stations ({time_from} to {time_to})')
ax.set_ylabel('Temperature (°F)')
ax.set_xlabel('Time')
fig.tight_layout()
plt.show()

In [ ]:
# plot a time series of wind speed for all the stations for a given time period
fig, ax = plt.subplots(figsize=(12, 5))
p = ds_merged['sknt'].sel(time=slice(time_from, time_to), station=ds_merged['station']).plot.line(x='time', hue='station', marker='x', ax=ax)
ax.set_title(f'Wind Speed Time Series for All Stations ({time_from} to {time_to})')
ax.set_ylabel('Wind Speed (knots)')
ax.set_xlabel('Time')
fig.tight_layout()
plt.show()

# plot a time series of maximum wind speed across all the stations for a given time period
fig, ax = plt.subplots(figsize=(12, 5))
p = ds_merged['sknt'].sel(time=slice(time_from, time_to), station=ds_merged['station']).max(dim='station').plot.line(x='time', marker='x')
ax.set_title(f'Maximum Wind Speed Across All Stations ({time_from} to {time_to})')
ax.set_ylabel('Wind Speed (knots)')
ax.set_xlabel('Time')
fig.tight_layout()
plt.show()

In [ ]:
# plot a time series of wind gust for all the stations for a given time period
fig, ax = plt.subplots(figsize=(12, 5))
p = ds_merged['gust'].sel(time=slice(time_from, time_to), station=ds_merged['station']).plot.line(x='time', hue='station', marker='x', ax=ax)
ax.set_title(f'Wind Gust Time Series for All Stations ({time_from} to {time_to})')
ax.set_ylabel('Wind Gust (knots)')
ax.set_xlabel('Time')
fig.tight_layout()
plt.show()

# plot a time series of maximum wind gust across all the stations for a given time period
fig, ax = plt.subplots(figsize=(12, 5))
p = ds_merged['gust'].sel(time=slice(time_from, time_to), station=ds_merged['station']).max(dim='station').plot.line(x='time', marker='x')
ax.set_title(f'Maximum Wind Gust Across All Stations ({time_from} to {time_to})')
ax.set_ylabel('Wind Gust (knots)')
ax.set_xlabel('Time')
fig.tight_layout()
plt.show()

In [ ]:
# plot a time series of precipitation for all the stations for a given time period
fig, ax = plt.subplots(figsize=(12, 5))
p = ds_merged['p01i'].sel(time=slice(time_from, time_to), station=ds_merged['station']).plot.line(x='time', hue='station', marker='x', ax=ax)
ax.set_title(f'Precipitation Time Series for All Stations ({time_from} to {time_to})')
ax.set_ylabel('Precipitation (inches)')
ax.set_xlabel('Time')
fig.tight_layout()
plt.show()

# plot a time series of maximum precipitation across all the stations for a given time period
fig, ax = plt.subplots(figsize=(12, 5))
p = ds_merged['p01i'].sel(time=slice(time_from, time_to), station=ds_merged['station']).max(dim='station').plot.line(x='time', marker='x')
ax.set_title(f'Maximum Precipitation Across All Stations ({time_from} to {time_to})')
ax.set_ylabel('Precipitation (inches)')
ax.set_xlabel('Time')
fig.tight_layout()
plt.show()

# Event Statistics

In [ ]:
def calculate_events_stats(dataset, event_threshold, station):
    # get a single dataframe merging customers_out, event_number_eaglei and maximum weather variables across all stations
    temp1 = dataset[['customers_out', 'event_number_eaglei']].to_dataframe().reset_index()
    if station == 'all':
        temp2 = dataset[['tmpf', 'sknt', 'gust', 'p01i','poccurence']].max(dim='station').to_dataframe().reset_index()
    else:
        dataset=dataset.sel(station=station)
        temp2 = dataset[['tmpf', 'sknt', 'gust', 'p01i','poccurence']].to_dataframe().reset_index()
    merged_df = pd.merge(temp1, temp2, on='time')

    merged_df_non_zero = merged_df[merged_df['customers_out'] > 0].copy().reset_index()
    event_numbers = merged_df_non_zero.event_number_eaglei.value_counts()[merged_df_non_zero.event_number_eaglei.value_counts()>=event_threshold].index.tolist()

    # import the function to calculate EAGLEi event stats
    from src.eaglei_modules.eagleiEventProcessing import get_eaglei_event_stats
    events_stats = get_eaglei_event_stats(merged_df_non_zero, event_numbers, event_method='eaglei', timestamp_column='time')

    # append weather stats to the events_stats dataframe
    weather_stats_list = []
    for event_num in event_numbers:
        event_data = merged_df_non_zero[merged_df_non_zero['event_number_eaglei'] == event_num]
        weather_stats = {
            'event_number': event_num,
            'max_sknt': event_data['sknt'].max(),
            'min_sknt': event_data['sknt'].min(),
            'avg_sknt': event_data['sknt'].mean(),

            'max_gust': event_data['gust'].max(),
            'min_gust': event_data['gust'].min(),
            'avg_gust': event_data['gust'].mean(),

            'max_tmpf': event_data['tmpf'].max(),
            'min_tmpf': event_data['tmpf'].min(),
            'avg_tmpf': event_data['tmpf'].mean(),

            'total_p01i': event_data['p01i'].sum(),
            'poccurence': event_data['poccurence'].max(),
            'max_p01i': event_data['p01i'].max(),
            'avg_p01i': event_data['p01i'].mean()
        }
        weather_stats_list.append(weather_stats)
    weather_stats_df = pd.DataFrame(weather_stats_list)
    events_stats = pd.merge(events_stats, weather_stats_df, on='event_number', how='left')

    return events_stats

events_stats = calculate_events_stats(ds_merged, event_threshold=30, station='all')

In [ ]:
events_stats

In [ ]:
# create an empirical ccdf of event sizes
sns.ecdfplot(data=events_stats, x='num_outages', complementary=True, color='red')
plt.xlabel('Event Size')
plt.ylabel('CCDF')
plt.title('Empirical CCDF of Event Sizes')
plt.show()

sns.ecdfplot(data=events_stats, x='num_outages', complementary=True, log_scale=True, color='red')
plt.xlabel('Event Size')
plt.ylabel('CCDF')
plt.yscale('log')
plt.title('Empirical CCDF of Event Sizes (Log-Log Scale)')
plt.show()

In [ ]:
# Basic Exponential Regression Model of Weather-Outage Relationship

from scipy.optimize import curve_fit

# Function to fit
def exponential_function(x,a,b,c):
    return a * np.exp(b*x) + c

def myround(x, base):
    return base * round(x/base)

weather_variable='total_p01i'
outage_variable='num_outages'

events_stats[weather_variable]=myround(events_stats[weather_variable],1)
# average all outage instances over their target weather variable
df_grouped = events_stats.groupby(weather_variable).agg({
    outage_variable: 'median'
}).reset_index()

# filter out anomalous weather variables that we do not have data to analyze
#df_grouped = df_grouped[df_grouped[weather_variable] >80]

# Raw Data
x = df_grouped[weather_variable].values
y = df_grouped[outage_variable].values
plt.scatter(x,y,label='Raw Binned Data')
#yscale('log')

# Fitted Data - May have to change p0 to fit the data
popt,pcov = curve_fit(exponential_function,x,y,p0=[10, 0.05, 0],maxfev=20000)
print(popt)
plt.plot(df_grouped[weather_variable], exponential_function(df_grouped[weather_variable], *popt), 'r-', label="Fitted Curve")

# Plot Settings
plt.title(f"Exponential Regression Model: {outage_variable} v.s. {weather_variable}")
plt.xlabel(f'{weather_variable}')
plt.ylabel(f'{outage_variable}')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Create correlation matrix between features in event_stats

x_cols=['max_sknt','max_gust','max_tmpf','min_tmpf','total_p01i','avg_p01i','poccurence']
y_cols=['duration_hours','max_customers_out','total_customers_out','num_outages','num_restores','customer_hours']

corr=events_stats[x_cols+y_cols].corr()

matrix=corr.loc[y_cols,x_cols]

plt.figure(figsize=(8,6))
sns.heatmap(matrix, annot=True, cmap="coolwarm", fmt=".2f", annot_kws={"size": 8},linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()